# 05 homr Confidence: Logits from Both Networks

homr is two neural networks with classical image processing in between:

```text
page ──► Segnet ──► per-pixel classes ──► staff detection ──► one image per staff
                                                                    │
                          tokens (rhythm, pitch, ...) ◄── TrOMR transformer
```

Both networks produce scores for every class, and homr keeps only the argmax.
`omr/homr_confidence.py` runs homr's own pipeline unchanged, but hooks the two
model calls so the full score vectors are kept:

```text
Segnet       6 class logits per pixel   -> softmax -> probabilities in U_image
TrOMR        6 heads per output token   -> softmax -> p, margin, entropy per head
             + one attention point      -> roughly where on the staff the token is
```

Things to keep in mind while reading the numbers:

- **The rhythm head never goes above about 0.95.** That looks like label
  smoothing during training. Use the *margin* (top-1 minus top-2) or the entropy
  rather than raw `p` for that head.
- **Decoding is greedy.** Each token's distribution is conditional on homr's own
  earlier choices, so a confident token after an earlier mistake can still be wrong.
- **Attention points are imprecise.** homr itself only uses them for debug drawings.
- **The segmentation probabilities are taken before homr's noise filtering.**

In [ ]:
from pathlib import Path
import sys
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap, Normalize

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["image.cmap"] = "gray"

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from omr.homr_confidence import load_homr_confidence, run_homr_with_confidence

# Same convention as notebook 04: dark orange = uncertain, pale = confident.
UNCERTAIN_CMAP = ListedColormap(plt.cm.Oranges(np.linspace(0.15, 0.95, 256)))
UNCERTAIN_CMAP.set_bad((0, 0, 0, 0))
CONFIDENT_CMAP = ListedColormap(UNCERTAIN_CMAP.colors[::-1])
PROB_CMAP = ListedColormap(plt.cm.Blues(np.linspace(0.0, 0.9, 256)))
BAR_COLOR = "#2a78d6"
INK_MUTED = "#52514e"


def faded(image, amount=0.55):
    return (255 - (255 - image.astype(np.float32)) * (1 - amount)).astype(np.uint8)

## 1. Run homr (or Load a Previous Run)

The first run takes about 20 seconds on CPU for one page. The results are
saved to `omr_output/homr/<stem>.confidence.json` and `.npz`, and later runs
just load them. Set `RERUN = True` to force a fresh run.

The same thing from the command line:

```bash
uv run python -m omr.homr_confidence example_sheet_music/springar.png -o omr_output/homr
```

In [ ]:
IMAGE_PATH = REPO / "example_sheet_music" / "springar.png"
OUT_DIR = REPO / "omr_output" / "homr"
RERUN = False

json_path = OUT_DIR / f"{IMAGE_PATH.stem}.confidence.json"
if json_path.exists() and not RERUN:
    conf = load_homr_confidence(json_path)
else:
    conf = run_homr_with_confidence(IMAGE_PATH, OUT_DIR)

page = cv2.imread(str(conf.image_path), cv2.IMREAD_GRAYSCALE)
homr_page = cv2.resize(page, conf.homr_shape[::-1], interpolation=cv2.INTER_AREA)
kept = [t for t in conf.tokens if t["kept"]]

print(f"original page {conf.original_shape}, homr page {conf.homr_shape}, crop offset {conf.crop_offset}")
print(f"segmentation probabilities: {conf.seg_probs.shape}  ({', '.join(conf.class_names)})")
print(f"{len(conf.staffs)} staves, {len(conf.tokens)} tokens ({len(kept)} kept by homr)")

## 2. Segmentation: Per-Pixel Probabilities

homr's Segnet labels every pixel as one of six classes. Here are the five
non-background probability maps for one staff. Change `STAFF` to move around.
The map lives in homr's working space, a 1920 px wide copy of the page.

In [ ]:
STAFF = 0

x0, y0, x1, y1 = (int(v) for v in conf.staffs[STAFF]["region_homr"])
x0, y0 = max(x0, 0), max(y0, 0)
classes = conf.class_names[1:]

fig, axes = plt.subplots(len(classes) + 1, 1, figsize=(14, 1.9 * (len(classes) + 1)), sharex=True)
axes[0].imshow(homr_page[y0:y1, x0:x1], vmin=0, vmax=255)
axes[0].set_title(f"staff {STAFF}: input", loc="left", fontsize=10)
for ax, name in zip(axes[1:], classes):
    plane = conf.seg_probs[conf.class_names.index(name), y0:y1, x0:x1]
    im = ax.imshow(plane, cmap=PROB_CMAP, vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"P({name})", loc="left", fontsize=10)
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes, fraction=0.015, pad=0.01, label="probability")
plt.show()

### Where does the segmentation hesitate?

The per-pixel uncertainty is the normalized entropy over the six classes:

```text
H(p) = -Σ p_c log p_c / log 6        0 = one class certain, 1 = all six equally likely
```

Most of the page is confidently background, so pixels with `H < MIN_SHOWN` are
left transparent.

In [ ]:
MIN_SHOWN = 0.05

probs = np.clip(conf.seg_probs, 1e-6, 1)
entropy = -(probs * np.log(probs)).sum(axis=0) / np.log(len(conf.class_names))
shown = np.where(entropy >= MIN_SHOWN, entropy, np.nan)

h, w = conf.homr_shape
fig, ax = plt.subplots(figsize=(13, 13 * h / w))
ax.imshow(faded(homr_page, 0.7), vmin=0, vmax=255)
im = ax.imshow(shown, cmap=UNCERTAIN_CMAP, vmin=0, vmax=0.6, interpolation="nearest")
ax.set_title("Segnet: normalized class entropy per pixel", loc="left")
ax.axis("off")
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label="normalized entropy")
plt.tight_layout()
plt.show()

print(f"pixels with entropy >= {MIN_SHOWN}: {(entropy >= MIN_SHOWN).mean():.2%}")

Which classes are being confused? For every uncertain pixel (top probability
below `MAX_TOP_P`) we count its two most likely classes.

In [ ]:
MAX_TOP_P = 0.6

order = np.argsort(conf.seg_probs, axis=0)
top1, top2 = order[-1], order[-2]
uncertain = conf.seg_probs.max(axis=0) < MAX_TOP_P
pairs = Counter(
    " / ".join(sorted((conf.class_names[a], conf.class_names[b])))
    for a, b in zip(top1[uncertain], top2[uncertain])
)

labels, counts = zip(*pairs.most_common(10))
fig, ax = plt.subplots(figsize=(9, 0.4 * len(labels) + 1))
ax.barh(labels[::-1], counts[::-1], color=BAR_COLOR, height=0.6)
ax.set_xlabel(f"pixels with top probability < {MAX_TOP_P}")
ax.set_title("most common top-2 class pairs among uncertain pixels", loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e6e6e3", lw=0.6)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 3. Transformer: Per-Token Confidence

The transformer reads one staff image (its *canvas*, 1280 x 256 px) and emits a
sequence of tokens. Every token has six heads:

```text
rhythm        note_4, rest_8, chord, clef_G2, keySignature_2, barline, ...
pitch         C4, D5, ... or "_" for no pitch
lift          accidental: #, b, N, or none
position      upper / lower staff of a grand staff
articulation  staccato, accent, ...
slur          slur / tie start or stop
```

For every head we stored the full probability vector (`conf.head_probs[head]`, one
row per token), plus a summary: top probability `p`, `margin` to the runner-up,
normalized entropy, and the top 3 alternatives.

**Two kinds of "nothing".** homr's vocabularies have two tokens that carry no
information on their own: `.` means "this token is not a note" (a barline, a clef,
a `chord` marker) and `_` means "a note without this decoration". A pitch or slur
head torn between `.` and `_` is only repeating the question "is this a note?",
which the rhythm head answers. Likewise, on a single (non-grand) staff homr treats
position `upper` the same as no position. So before taking margins, we merge
tokens that mean the same thing:

```text
grouped margin = P(most likely meaning) − P(second most likely meaning)
```

In [ ]:
HEADS = list(conf.vocabs)
EMPTY = {".", "_"}  # "not a note" / "note without this decoration"


def meaning(head, name, single_staff):
    if name in EMPTY and head != "rhythm":
        return "nothing"
    if head == "position" and single_staff and name == "upper":
        return "nothing"
    return name


def grouped_margin(tok, head):
    single_staff = not conf.staffs[tok["staff"]]["is_grandstaff"]
    groups = Counter()
    for name, p in zip(conf.vocabs[head], conf.head_probs[head][tok["index"]]):
        groups[meaning(head, name, single_staff)] += float(p)
    top = sorted(groups.values(), reverse=True) + [0.0]
    return float(np.clip(top[0] - top[1], 0, 1))  # clip: stored float16 sums can exceed 1 slightly


MARGIN = {head: np.array([grouped_margin(t, head) for t in conf.tokens]) for head in HEADS}

How uncertain is each head, across all tokens on the page?

In [ ]:
fig, axes = plt.subplots(1, len(HEADS), figsize=(14, 2.8), sharey=True)
bins = np.linspace(0, 1, 26)
for ax, head in zip(axes, HEADS):
    margins = [MARGIN[head][t["index"]] for t in kept]
    ax.hist(margins, bins=bins, color=BAR_COLOR, rwidth=0.85)
    ax.set_yscale("log")
    ax.set_title(head, fontsize=10)
    ax.set_xlabel("grouped margin", fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("tokens (log scale)")
plt.tight_layout()
plt.show()

To get one number per token we take the **weakest head**: the smallest grouped
margin across the heads. A token is only as trustworthy as its least certain part.
Change `token_score` to try other summaries, for example rhythm and pitch only.

In [ ]:
def token_score(tok, heads=HEADS):
    return min(MARGIN[h][tok["index"]] for h in heads)


def weakest_head(tok, heads=HEADS):
    return min(heads, key=lambda h: MARGIN[h][tok["index"]])


def describe(tok):
    rhythm, pitch, lift = (tok["heads"][h]["token"] for h in ("rhythm", "pitch", "lift"))
    pitch = "" if pitch in EMPTY else f" {pitch}"
    lift = "" if lift in EMPTY else lift
    return f"{rhythm}{pitch}{lift}"


SCORE_NORM = Normalize(0, 1)

print(f"{'staff':>5} {'step':>4}  {'token':<24}{'score':>6}  weakest head: top alternatives")
for tok in sorted(kept, key=token_score)[:15]:
    head = weakest_head(tok)
    alts = ", ".join(f"{name} {p:.2f}" for name, p in tok["heads"][head]["top"])
    print(f"{tok['staff']:>5} {tok['step']:>4}  {describe(tok):<24}{token_score(tok):>6.2f}  {head}: {alts}")

### Tokens on the staff the model actually saw

Each kept token is drawn at its attention point on the canvas, coloured by
`token_score` (dark = uncertain). Only the uncertain ones (score below
`LABEL_BELOW`) get a label, so the picture stays readable. `chord` tokens, which
only mean "the next note sounds together with the previous one", are left out.

In [ ]:
STAFF = 0
LABEL_BELOW = 0.5

canvas = conf.canvases[STAFF]
tokens = [t for t in kept if t["staff"] == STAFF and t["heads"]["rhythm"]["token"] != "chord"]

fig, ax = plt.subplots(figsize=(16, 16 * canvas.shape[0] / canvas.shape[1] + 0.6))
ax.imshow(faded(canvas, 0.3), vmin=0, vmax=255)
for tok in tokens:
    u, v = tok["canvas_xy"]
    if not (np.isfinite(u) and np.isfinite(v)):
        continue
    score = token_score(tok)
    ax.scatter(u, v, s=70, color=CONFIDENT_CMAP(SCORE_NORM(score)), edgecolor="white", linewidth=1, zorder=3)
    if score < LABEL_BELOW:
        ax.annotate(f"{tok['step']}: {describe(tok)}", (u, v), xytext=(4, -8), textcoords="offset points",
                    fontsize=8, color="#0b0b0b")
ax.set_title(f"staff {STAFF}: {len(tokens)} tokens at their attention points, coloured by token score", loc="left")
ax.axis("off")
fig.colorbar(plt.cm.ScalarMappable(SCORE_NORM, CONFIDENT_CMAP), ax=ax, fraction=0.015, pad=0.01, label="token score")
plt.tight_layout()
plt.show()

## 4. Lower the Tokens onto the Page

Each token's attention point is mapped back to original-page pixels (through
homr's crop and scale; per-staff dewarping is ignored). That puts homr's token
confidence in the same `U_image` coordinates as Audiveris's symbol boxes in
notebook 04, so the two can later be overlaid.

First on the page:

In [ ]:
oh, ow = conf.original_shape
fig, ax = plt.subplots(figsize=(13, 13 * oh / ow))
ax.imshow(faded(page), vmin=0, vmax=255)
for staff in conf.staffs:
    bx0, by0, bx1, by1 = staff["box_original"]
    ax.add_patch(plt.Rectangle((bx0, by0), bx1 - bx0, by1 - by0, fill=False, edgecolor=INK_MUTED, lw=0.6, ls="--"))
pts = [(t["original_xy"], token_score(t)) for t in kept
       if t["original_xy"][0] is not None and t["heads"]["rhythm"]["token"] != "chord"]
xy = np.array([p for p, _ in pts])
scores = np.array([s for _, s in pts])
order = np.argsort(-scores)  # uncertain tokens drawn last, on top
ax.scatter(xy[order, 0], xy[order, 1], c=scores[order], cmap=CONFIDENT_CMAP, norm=SCORE_NORM,
           s=40, edgecolor="white", linewidth=0.8)
ax.set_title(f"{IMAGE_PATH.name}: homr tokens coloured by token score (dashed: detected staves)", loc="left")
ax.axis("off")
fig.colorbar(plt.cm.ScalarMappable(SCORE_NORM, CONFIDENT_CMAP), ax=ax, fraction=0.025, pad=0.01, label="token score")
plt.tight_layout()
plt.show()

### Per-staff confidence along `x`

The same shape as the Audiveris profile in notebook 04: one row per staff, `x`
in original-page pixels, one stem per token. Because the attention points jitter,
the `x` of a single token is only approximate.

In [ ]:
fig, axes = plt.subplots(len(conf.staffs), 1, figsize=(14, 1.1 * len(conf.staffs)), sharex=True, squeeze=False)
for ax, staff in zip(axes[:, 0], conf.staffs):
    toks = [t for t in kept if t["staff"] == staff["index"] and t["original_xy"][0] is not None
            and t["heads"]["rhythm"]["token"] != "chord"]
    xs = np.array([t["original_xy"][0] for t in toks])
    ys = np.array([token_score(t) for t in toks])
    ax.vlines(xs, 0, ys, color=BAR_COLOR, lw=1.2)
    ax.scatter(xs, ys, s=12, color=BAR_COLOR, zorder=3)
    bx0, _, bx1, _ = staff["box_original"]
    ax.axvspan(0, bx0, color="#f0f0ed", lw=0)
    ax.axvspan(bx1, ow, color="#f0f0ed", lw=0)
    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.5, 1])
    ax.tick_params(labelsize=7)
    ax.set_ylabel(f"staff {staff['index']}", rotation=0, ha="right", va="center", fontsize=8)
    ax.text(ow, 0.5, f" {len(toks)} tokens", va="center", fontsize=7, color=INK_MUTED)
axes[-1, 0].set_xlim(0, ow)
axes[-1, 0].set_xlabel("x (original page pixels)")
fig.suptitle("homr token score along each staff", x=0.01, ha="left")
plt.tight_layout()
plt.show()

## 5. What to Look For

- Do the segmentation's uncertain pixels sit on symbol edges only (expected), or
  are there whole objects where it cannot decide (notehead vs symbol, stem vs
  staff)?
- Staff 3 on `springar.png` is a staff homr found in the bleed-through from the
  reverse page, next to the title. Do its tokens stand out as uncertain, and does
  the segmentation entropy light up there too?
- Which head makes tokens uncertain? Usually rhythm, and its alternatives (chord
  vs rest, note_16 vs note_24) say what kind of ambiguity it is.
- Are the uncertain tokens the same places where Audiveris had low grades in
  notebook 04? Both now live in `U_image`.

Possible next steps:

```text
fuse:     overlay Audiveris symbol boxes and homr token points on one staff
match:    pair Audiveris noteheads with homr note tokens by x and pitch
compare:  agreement + confidence from both -> a joint uncertainty per event
```